# OpenData UK — our Horizon vs. original Horizon (§6.1)

Runs our Horizon implementation (`horizon/horizon.py`) on the three
`datasets/opendatauk/` example tables, then compares its repair quality
(precision / recall / F1) against the repair produced by the **original**
Horizon implementation (`horizon_ref.csv`).

For each table, three row-aligned tables are scored cell-by-cell (all read as
`Utf8` so values compare like-for-like):

- **clean** — ground truth (`clean.csv`)
- **dirty** — BART-injected errors fed to the repairer (`injected/e3_r10.csv`,
  error type `e3` = `TypoRandom`, rate 10%)
- **cleaned** — the repair output: *ours* (`output/<table>_cleaned_data.csv`,
  written by the Horizon CLI) vs. the *reference* (`horizon_ref.csv`)

Both repairs are scored against the same `clean`/`dirty` pair via the single
`eval.effectiveness_eval.evaluate_repair` entry point, so `n_dirty` is shared
and only `n_repaired` / `n_correct` differ between the two repairers.

In [5]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
import polars as pl

REPO = Path("..").resolve()
# repo root on sys.path so `eval` resolves from notebooks/
sys.path.insert(0, str(REPO))

from eval.effectiveness_eval import evaluate_repair

In [6]:
# the three example tables, the injected error file to repair, and where outputs go
TABLES = ["uk_csv_127_263", "uk_csv_227_5k", "uk_csv_270_9k"]
ERRORTYPE = "e3"
RATE = 10
DIRTY_REL = f"injected/{ERRORTYPE}_r{RATE:02d}.csv"  # relative to each dataset dir
REF_FILE = "horizon_ref.csv"

DATASETS = REPO / "datasets" / "opendatauk"
OUTPUT = REPO / "output"
HORIZON = REPO / "horizon" / "horizon.py"

## Run our Horizon

Invoke the CLI once per table to repair `injected/e3_r10.csv`. The cleaned table
is written to `output/<table>_cleaned_data.csv`.

In [7]:
def run_horizon(name: str) -> Path:
    """Run the Horizon CLI on one table; return the cleaned-output path.

    horizon.py uses flat imports, so it must run with horizon/ as the script
    dir (Python puts it on sys.path[0] automatically when invoked by file).
    """
    ds_dir = DATASETS / name
    proc = subprocess.run(
        [sys.executable, str(HORIZON),
         "-ds", str(ds_dir), "-dd", DIRTY_REL, "-o", str(OUTPUT), "-l", "ERROR"],
        cwd=str(REPO), capture_output=True, text=True,
    )
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError(f"Horizon CLI failed on {name}")
    return OUTPUT / f"{name}_cleaned_data.csv"


for name in TABLES:
    out = run_horizon(name)
    print(f"{name}: wrote {out.name}")

uk_csv_127_263: wrote uk_csv_127_263_cleaned_data.csv
uk_csv_227_5k: wrote uk_csv_227_5k_cleaned_data.csv
uk_csv_270_9k: wrote uk_csv_270_9k_cleaned_data.csv


## Score both repairs

Read clean / dirty / cleaned / reference as `Utf8`, then score our output and
the reference repair against the same ground truth with `evaluate_repair`.

In [8]:
def load_utf8(path: Path) -> pl.DataFrame:
    """Read a table as all-Utf8 with lowercased column names."""
    df = pl.read_csv(path, infer_schema_length=0)
    return df.rename({c: c.strip().lower() for c in df.columns})


rows = []
for name in TABLES:
    ds_dir = DATASETS / name
    clean = load_utf8(ds_dir / "clean.csv")
    dirty = load_utf8(ds_dir / DIRTY_REL)
    ours = load_utf8(OUTPUT / f"{name}_cleaned_data.csv")
    ref = load_utf8(ds_dir / REF_FILE)

    m_ours = evaluate_repair(clean, dirty, ours)
    m_ref = evaluate_repair(clean, dirty, ref)

    rows.append(
        {
            "dataset": name,
            "n_dirty": m_ours["n_dirty"],
            "our_repaired": m_ours["n_repaired"],
            "our_correct": m_ours["n_correct"],
            "our_P": m_ours["precision"],
            "our_R": m_ours["recall"],
            "our_F1": m_ours["f1"],
            "ref_repaired": m_ref["n_repaired"],
            "ref_correct": m_ref["n_correct"],
            "ref_P": m_ref["precision"],
            "ref_R": m_ref["recall"],
            "ref_F1": m_ref["f1"],
        }
    )
    print(
        f"{name}: ours F1={m_ours['f1']:.3f} (P={m_ours['precision']:.3f} "
        f"R={m_ours['recall']:.3f})  |  ref F1={m_ref['f1']:.3f} "
        f"(P={m_ref['precision']:.3f} R={m_ref['recall']:.3f})"
    )

uk_csv_127_263: ours F1=0.104 (P=0.082 R=0.142)  |  ref F1=0.092 (P=0.087 R=0.097)
uk_csv_227_5k: ours F1=0.211 (P=0.187 R=0.242)  |  ref F1=0.224 (P=0.211 R=0.238)
uk_csv_270_9k: ours F1=0.230 (P=0.259 R=0.207)  |  ref F1=0.222 (P=0.293 R=0.178)


## Comparison

In [9]:
df = pd.DataFrame(rows)
metric_cols = ["our_P", "our_R", "our_F1", "ref_P", "ref_R", "ref_F1"]
df[metric_cols] = df[metric_cols].round(3)

# head-to-head P/R/F1 plus our advantage over the reference repair
compare = df[["dataset", "our_P", "ref_P", "our_R", "ref_R", "our_F1", "ref_F1"]].copy()
compare["dF1 (ours - ref)"] = (df["our_F1"] - df["ref_F1"]).round(3)
compare

,dataset,our_P,ref_P,our_R,ref_R,our_F1,ref_F1,dF1 (ours - ref)
0,uk_csv_127_263,0.082,0.087,0.142,0.097,0.104,0.092,0.012
1,uk_csv_227_5k,0.187,0.211,0.242,0.238,0.211,0.224,-0.013
2,uk_csv_270_9k,0.259,0.293,0.207,0.178,0.230,0.222,0.008


In [10]:
# F1 head-to-head per table
ax = (
    df.set_index("dataset")[["our_F1", "ref_F1"]]
    .plot.bar(rot=0, figsize=(7, 4))
)
ax.set_ylabel("F1")
ax.set_ylim(0, 1)
ax.set_title("Repair F1: our Horizon vs. original Horizon (e3, r10)")
ax.legend(["ours", "reference"])
ax.figure.tight_layout()

In [11]:
# raw cell counts behind the metrics (n_dirty is shared by both repairers)
df[
    [
        "dataset",
        "n_dirty",
        "our_repaired",
        "our_correct",
        "ref_repaired",
        "ref_correct",
    ]
]

,dataset,n_dirty,our_repaired,our_correct,ref_repaired,ref_correct
0,uk_csv_127_263,226,392,32,252,22
1,uk_csv_227_5k,5451,7059,1321,6147,1298
2,uk_csv_270_9k,16630,13298,3446,10111,2963


## Error detectability — why both F1 scores are low, and why so many cells get touched

An FD repairer can only act on errors that create a **violation**: two rows that
share an LHS value but disagree on the RHS. Errors anywhere else are invisible to
the FDs. Two things follow, both visible in the table below:

- **Recall ceiling (`detectable` / `max_recall`)** — errors that never enter a
  violation cannot be repaired by *any* FD method, so recall is capped well below
  1.0 before repair quality even matters. Compare `max_recall` to the achieved
  `our_R` / `ref_R`.
- **`cells_flagged` ≫ `errors`** — FDs flag *every* cell in a violating group, not
  just the wrong one. `correct_flagged` counts the **correct** cells dragged into
  violations — mostly via the only redundant FDs (`region_code ↔ region_name`,
  whose groups span ~26 rows, so one bad cell flags its whole class). The repairer
  rewrites cells across these groups, so it touches far more cells than there are
  errors (inflating `n_repaired`) and sinks precision every time it picks a
  non-original value for a cell that was already correct.

In [ ]:
def detectable_cells(dirty: pl.DataFrame, fds: pl.DataFrame) -> set:
    """(row, col) cells that sit in an FD violation in `dirty`.

    A cell is detectable iff its row belongs to a group of rows that share an
    LHS value but disagree on the RHS. Both the disagreeing RHS cell and the
    LHS cell(s) of such a group are flagged. Errors outside any violation are
    invisible to the FDs, so no FD repairer can fix them.
    """
    idx = pl.Series("__i", range(dirty.height))
    cells: set = set()
    for lhs, rhs in fds.iter_rows():
        lcols = [a.strip().lower() for a in lhs.split(";")]
        rhs = rhs.lower()
        bad = (
            dirty.with_columns(idx)
            .group_by(lcols)
            .agg(pl.col(rhs).n_unique().alias("u"), pl.col("__i").alias("rows"))
            .filter(pl.col("u") > 1)
        )
        for group_rows in bad["rows"].to_list():
            for i in group_rows:
                cells.add((i, rhs))
                for lc in lcols:
                    cells.add((i, lc))
    return cells


det_rows = []
for name in TABLES:
    ds_dir = DATASETS / name
    clean = load_utf8(ds_dir / "clean.csv")
    dirty = load_utf8(ds_dir / DIRTY_REL)
    fds = pl.read_csv(ds_dir / "fds.csv")

    cols = [c for c in clean.columns if c in dirty.columns]
    errors = {
        (i, c) for c in cols for i in range(clean.height) if dirty[c][i] != clean[c][i]
    }
    flagged = detectable_cells(dirty, fds)
    det_err = errors & flagged

    det_rows.append(
        {
            "dataset": name,
            "errors": len(errors),
            "detectable": len(det_err),
            "undetectable": len(errors) - len(det_err),
            "max_recall": round(len(det_err) / max(1, len(errors)), 3),
            "cells_flagged": len(flagged),
            "correct_flagged": len(flagged - errors),
        }
    )

det = pd.DataFrame(det_rows).merge(df[["dataset", "our_R", "ref_R"]], on="dataset")
det